# Конспект к задаче 2

## 1. Постановка задачи

Решается задача Коши для скалярного ОДУ

$$
y'(t)=f(t,y), \qquad y(t_0)=y_0.
$$

В программе для задачи 2 в `main.py` сейчас задан пример

$$
y' = y+t, \qquad y(0)=1,
$$

с точным решением

$$
y(t)=Ce^t-t-1, \qquad C=y_0+t_0+1.
$$

Вычисления ведутся на равномерной сетке

$$
t_n=t_0+nh.
$$

Используется неявный 4-шаговый метод Адамса-Моултона порядка 5 с явным предиктором Адамса-Башфорта 4-го порядка.

## 2. Идея метода

### 2.1. Почему метод многошаговый

Метод Адамса использует значения правой части не только в текущем узле, но и в нескольких предыдущих узлах. За счет этого можно получить высокий порядок точности без вычисления производных высоких порядков.

Минус такого подхода: метод **не самостартующий**, то есть ему нужны стартовые значения на первых шагах.

### 2.2. Предиктор AB4

Сначала строится прогноз следующего значения:

$$
y_{n+1}^{(0)} = y_n + \frac{h}{24}
\left(
55f_n - 59f_{n-1} + 37f_{n-2} - 9f_{n-3}
\right),
$$

где

$$
f_k=f(t_k,y_k).
$$

Это явный 4-шаговый метод Адамса-Башфорта. Он дает хорошее начальное приближение для неявного шага.

### 2.3. Корректор AM4

Далее используется неявная формула Адамса-Моултона:

$$
y_{n+1}=y_n+\frac{h}{720}
\left(
251f_{n+1}+646f_n-264f_{n-1}+106f_{n-2}-19f_{n-3}
\right),
$$

где

$$
f_{n+1}=f(t_{n+1},y_{n+1}).
$$

Неявность в том, что $y_{n+1}$ входит в правую часть через $f_{n+1}$.

### 2.4. Как снимается неявность

В программе используется метод простых итераций:

$$
y^{(m+1)} = y_n + \frac{h}{720}
\left(
251f(t_{n+1},y^{(m)})+646f_n-264f_{n-1}+106f_{n-2}-19f_{n-3}
\right).
$$

Начальное приближение $y^{(0)}$ берется из предиктора AB4. Итерации останавливаются, когда

$$
|y^{(m+1)}-y^{(m)}| < \text{tol},
$$

или достигнут лимит `max_iter`. Если сходимости нет, программа ставит флаг/предупреждение, но не падает.

## 3. Старт метода

Так как метод 4-шаговый, для запуска нужны значения

$$
y_0,\; y_1,\; y_2,\; y_3.
$$

Значение $y_0$ задано в условии, а $y_1,y_2,y_3$ вычисляются методом Рунге-Кутты 4-го порядка (RK4).

Это важно на защите: метод Адамса-Моултона сам по себе стартовать не может, поэтому первые три шага получаем одноступенчатым методом.

## 4. Точность, сходимость, ограничения

### 4.1. Порядок точности

Для 4-шагового Adams-Moulton:

- локальная погрешность одного шага: $\tau_{n+1}=O(h^6)$;
- глобальная погрешность на всем отрезке: $e_n=O(h^5)$.

Это означает, что при уменьшении шага в 2 раза в асимптотическом режиме ошибка должна уменьшаться примерно в

$$
2^5=32
$$

раз.

Предиктор AB4 имеет порядок 4, но он используется только как начальное приближение для итераций, а окончательная формула остается порядка 5.

### 4.2. Почему метод сходится

Для линейных многшаговых методов нужны согласованность и нулевая устойчивость. Для Adams-Moulton эти свойства выполнены, поэтому метод сходится на гладких задачах.

Практически это значит: если $f$ достаточно гладкая, а шаг $h$ достаточно мал, то численное решение стремится к точному при $h \to 0$.

### 4.3. Ограничения метода

1. Метод рассчитан на достаточно гладкую правую часть $f(t,y)$.
2. Метод плохо подходит для жестких задач в варианте с простыми итерациями.
3. Для сходимости fixed-point итераций полезно, чтобы отображение было сжимающим. Грубо:

$$
\frac{251}{720}hL < 1,
$$

где $L$ — константа Липшица по $y$.
4. Метод использует **постоянный шаг**.
5. Если последний шаг не помещается точно в сетку, в программе он добирается методом RK4 укороченным шагом.

### 4.4. Влияние параметров `h`, `tol`, `max_iter`

- `h` влияет на дискретизационную погрешность: основной вклад порядка $O(h^5)$;
- `tol` влияет на точность решения неявного шага;
- `max_iter` ограничивает число итераций на одном шаге.

Итоговая ошибка на практике состоит из

$$
\text{ошибка} \approx O(h^5) + \text{ошибка итерационного решения}.
$$

## 5. Как устроена программа

### 5.1. Архитектура

Задача разбита на модули:

- `solver.py` содержит `rk4_step` и `solve_adams_moulton4`;
- `input_parser.py` читает `in.txt`;
- `csv_io.py` пишет CSV-файлы;
- `plotter.py` строит `comparison.png`;
- `main.py` задает `f`, `exact` и запускает весь сценарий.

### 5.2. Что происходит по шагам

1. Из `in.txt` читаются `t0`, `y0`, `t_end`, список шагов `h_values`, `exact_h`, `tol`, `max_iter`.
2. В `main.py` задаются функция правой части `f(t,y)` и точное решение `exact(t,t0,y0)`.
3. На частой сетке строится точное решение и записывается в `exact_xy.csv`.
4. Для каждого шага из `h_values` решается задача методом Adams-Moulton, создаются `num_h_*.csv` и `num_vs_exact_h_*.csv`, считаются `mean_abs_error` и `max_abs_error`.
5. По всем сериям строится общий график `comparison.png`.
6. В `summary.csv` собирается итоговая сводка по каждому шагу.

## 6. Что показывают выходные файлы

- `exact_xy.csv` — точное решение;
- `num_h_*.csv` — численное решение для конкретного шага;
- `num_vs_exact_h_*.csv` — сравнение по узлам: `x`, `y_num`, `y_exact`, `delta = |y_num-y_exact|`;
- `summary.csv` — средняя и максимальная ошибка;
- `comparison.png` — график точного и численных решений.

## 7. Коротко, что говорить преподавателю

1. Решается задача Коши для ОДУ на равномерной сетке.
2. Основной метод — неявный 4-шаговый Adams-Moulton порядка 5.
3. Для снятия неявности используется predictor-corrector: предиктор AB4 и корректор AM4 с простыми итерациями.
4. Метод не самостартующий, поэтому первые 3 шага считаются RK4.
5. Теоретически: локальная ошибка $O(h^6)$, глобальная ошибка $O(h^5)$, при делении шага пополам ошибка должна убывать примерно в 32 раза.
6. Ограничения: нужна гладкая функция, для жестких задач и больших шагов простые итерации могут не сходиться.
7. В программе результат сравнивается с точным решением, записывается в CSV и строится график.

## 8. Одной фразой итог

В задаче 2 реализован классический predictor-corrector на базе AB4 + AM4, который дает высокую точность порядка 5 на гладких не-жестких задачах и позволяет удобно сравнить численное решение с точным по таблицам и графикам.